In [ ]:
# Instalação das bibliotecas;

In [1]:
import json
import os
import tempfile
import time
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import pystac_client
import rasterio
import seaborn as sns

/opt/conda/envs/geospatial/lib/python3.11/site-packages/seaborn/_statistics.py:32: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.stats import gaussian_kde


In [ ]:
# Prepara o script para não travar a execução por erro de conexão

In [2]:
os.environ['GDAL_HTTP_MAX_RETRY'] = '5'
os.environ['GDAL_HTTP_RETRY_DELAY'] = '3'

In [ ]:
# Acesso ao servidor de STAC, para ler as coleções, as cenas e as bandas definidas;

# Indicação das pastas de saída para guardar as tabelas csv, json e os gráficos ;

# Definição dos valores dos parâmetros para executar a compressão do arquivo gtiff;

In [3]:
STAC_URL = "https://data.inpe.br/bdc/stac/v1/"
COLLECTION_ID = 'S2-16D-2'
SCENE_IDS = [
    "S2-16D_V2_002011_20260728",
    "S2-16D_V2_001014_20260728",
    "S2-16D_V2_002012_20260728",
]
BANDS = ['B01','B02','B03','B04','B05','B06','B07','B08','B09','B11','B12','B8A','EVI','NBR','NDVI','SCL']

OUTPUT_DIR = Path('/home/jovyan/Downloads/stac_benchmark_gtiff_zstdbeta10')
METRICS_DIR = OUTPUT_DIR / 'metricas_json'

# Grid ZSTD (Níveis de compressão 1 a 15 e Preditores 1=Nenhum, 2=Horizontal, 3=Float)
ZSTD_CONFIGS = [
    {'level': 1,  'predictor': 2, 'label': 'ZSTD_L01_P2'},
    {'level': 3,  'predictor': 2, 'label': 'ZSTD_L03_P2'},
    {'level': 5,  'predictor': 2, 'label': 'ZSTD_L05_P2'},
    {'level': 7,  'predictor': 2, 'label': 'ZSTD_L07_P2'},
    {'level': 9,  'predictor': 2, 'label': 'ZSTD_L09_P2'},
    {'level': 11, 'predictor': 2, 'label': 'ZSTD_L11_P2'},
    {'level': 13, 'predictor': 2, 'label': 'ZSTD_L13_P2'},
    {'level': 15, 'predictor': 2, 'label': 'ZSTD_L15_P2'},
]

In [ ]:
# Cria a pasta onde ficarão os arquivos que serão gerados, garante a criação de um diretório no sistema de arquivos.

In [4]:
def garante_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

In [ ]:
# Função que faz a verificação para evitar o problema de muitas requisições consecutivas ao servidor;

In [5]:
def stac_search_with_retry(client, collection_id: str, scene_id: str, max_retries: int = 5) -> list:
    for attempt in range(max_retries):
        try:
            search = client.search(collections=[collection_id], ids=[scene_id])
            return list(search.items())
        except Exception as e:
            if "429" in str(e) or "Too Many Requests" in str(e):
                wait_time = (2 ** attempt) + 3
                print(f"   [RATE LIMIT 429] Aguardando {wait_time}s (Tentativa {attempt + 1}/{max_retries})...")
                time.sleep(wait_time)
            else:
                raise e
    raise Exception(f"Falha ao buscar a cena {scene_id} após {max_retries} tentativas.")

In [ ]:
# Cria uma pasta temporária, onde serão armazenados os arquivos gtiff comprimidos que serão descartados após o fim da execução do script;

# Cria uma cópia limpa das propriedades do raster (dimensões, tipo de dado, sistema de coordenadas, blocagem) para que os testes
#não sobrescrevam a base original;
 
# Converte para MB;

# Faz o cálculo do tempo que levou para o arquivo RAW ser lido e apesar de não ter sido comprimido, armazena as informações de tamanho final,
#taxa de compressão, tempo de compressão e tempo de leitura;

# Na sequência, faz a compressão do arquivo RAW conforme parâmetros selecionados (Deflate, LZW ou ZSTD) e também mede o desempenho de velocidade
#do algoritmo;

# Guarda as informações após compressão para depois gerar os gráficos e tabelas;

In [6]:
def evaluate_asset_in_temp(asset_href: str) -> list[dict]:
    results = []
    with tempfile.TemporaryDirectory() as temp_dir_str:
        temp_dir = Path(temp_dir_str)
        raw_path = temp_dir / "raw.tif"

        with rasterio.open(asset_href) as src:
            profile = src.profile.copy()
            profile.update({
                'driver': 'GTiff',
                'compress': None,
                'tiled': True,
                'blockxsize': 256,
                'blockysize': 256,
            })
            data = src.read()
            with rasterio.open(raw_path, 'w', **profile) as dst:
                dst.write(data)

        raw_size_bytes = raw_path.stat().st_size
        raw_size_mb = raw_size_bytes / (1024 * 1024)

        t0_read = time.time()
        with rasterio.open(raw_path) as src:
            _ = src.read()
        raw_read_time = time.time() - t0_read

        results.append({
            'Algoritmo': 'RAW_SemComp',
            'Level': 0,
            'Predictor': 0,
            'Tamanho_Final_MB': round(raw_size_mb, 2),
            'Taxa_Compressao (%)': 0.0,
            'Tempo_Compressao_segundos': 0.0,
            'Tempo_Leitura_segundos': round(raw_read_time, 3),
        })

        for cfg in ZSTD_CONFIGS:
            comp_path = temp_dir / f"{cfg['label']}.tif"
            comp_profile = profile.copy()
            comp_profile.update({
                'compress': 'zstd',
                'zstd_level': cfg['level'],
                'predictor': cfg['predictor'],
                'tiled': True,
                'blockxsize': 256,
                'blockysize': 256,
            })

            start_write = time.time()
            with rasterio.open(comp_path, 'w', **comp_profile) as dst:
                dst.write(data)
            time_write = time.time() - start_write

            start_read = time.time()
            with rasterio.open(comp_path) as src_comp:
                _ = src_comp.read()
            time_read = time.time() - start_read

            comp_size_bytes = comp_path.stat().st_size
            comp_size_mb = comp_size_bytes / (1024 * 1024)
            reduction = (1 - (comp_size_bytes / raw_size_bytes)) * 100

            results.append({
                'Algoritmo': cfg['label'],
                'Level': cfg['level'],
                'Predictor': cfg['predictor'],
                'Tamanho_Final_MB': round(comp_size_mb, 2),
                'Taxa_Compressao (%)': round(reduction, 2),
                'Tempo_Compressao_segundos': round(time_write, 3),
                'Tempo_Leitura_segundos': round(time_read, 3),
            })

    return results

In [ ]:
# Exibe a execução do processamento de cada cena, salvando as métricas em tabela json, caso a cena já foi processada, não perde tempo
#processando novamente agilizando o processo;

In [7]:
def executar_fase_processamento():
    garante_dir(OUTPUT_DIR)
    garante_dir(METRICS_DIR)
    print(f'Conectando ao STAC: {STAC_URL}...')
    client = pystac_client.Client.open(STAC_URL)

    for scene_id in SCENE_IDS:
        print('\n' + '=' * 50)
        print(f'Processando Cena: {scene_id}')
        print('=' * 50)
        items = stac_search_with_retry(client, COLLECTION_ID, scene_id)
        if not items:
            continue
        item = items[0]

        for band_name in BANDS:
            json_metric_path = METRICS_DIR / f'metricas_{scene_id}_{band_name}.json'
            if json_metric_path.exists():
                print(f'   [CACHE] JSON já existe: {json_metric_path.name}')
                continue
            if band_name not in item.assets:
                continue

            print(f'--> Métricas GeoTIFF ZSTD para Banda: {band_name}')
            asset_href = item.assets[band_name].href
            try:
                metrics = evaluate_asset_in_temp(asset_href)
                for metric in metrics:
                    metric['Cena'] = scene_id
                    metric['Banda'] = band_name
                with open(json_metric_path, 'w', encoding='utf-8') as f:
                    json.dump(metrics, f, indent=4, ensure_ascii=False)
                print(f'   [MÉTRICAS SALVAS] {json_metric_path.name}')
            except Exception as e:
                print(f'   [ERRO] {e}')
            time.sleep(2)

In [ ]:
# Exibe a análise comparativa e as métricas salvas;

# Salva a planilha criada em uma arquivo csv;

# Gera os gráficos Tamanho Médio após Compressão (MB), Taxa Média de Compressão (%), Tempo Médio para Compressão (s) e Tempo Médio de Leitura
#após Compressão (s) com os valores númericos para facilitar a leitura;

In [8]:
def executar_fase_analise():
    print('\n' + '=' * 50)
    print('    ANÁLISE COMPARATIVA: GTIFF ZSTD')
    print('=' * 50)
    json_files = list(METRICS_DIR.glob('*.json'))
    if not json_files:
        print('Nenhum JSON encontrado.')
        return

    todas_metricas = []
    for jf in json_files:
        with open(jf, 'r', encoding='utf-8') as f:
            todas_metricas.extend(json.load(f))

    df = pd.DataFrame(todas_metricas)
    csv_consolidado = OUTPUT_DIR / 'planilha_gtiff_zstd.csv'
    df.to_csv(csv_consolidado, index=False)
    print(f'CSV salvo em: {csv_consolidado}')

    sns.set_theme(style='whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))

    # 1. Tamanho Médio após Compressão (MB)
    ax1 = axes[0, 0]
    sns.barplot(data=df, x='Algoritmo', y='Tamanho_Final_MB', hue='Algoritmo', ax=ax1, palette='viridis', errorbar=None, legend=False)
    ax1.set_title('GeoTIFF ZSTD: Tamanho Médio após Compressão (MB)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('')
    ax1.set_ylabel('Tamanho (MB)')
    ax1.tick_params(axis='x', rotation=45)

    # 2. Taxa Média de Compressão (%)
    ax2 = axes[0, 1]
    sns.barplot(data=df, x='Algoritmo', y='Taxa_Compressao (%)', hue='Algoritmo', ax=ax2, palette='crest', errorbar=None, legend=False)
    ax2.set_title('GeoTIFF ZSTD: Taxa Média de Compressão (%)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('')
    ax2.set_ylabel('Taxa (%)')
    ax2.tick_params(axis='x', rotation=45)

    # 3. Tempo Médio para Compressão (s)
    ax3 = axes[1, 0]
    sns.barplot(data=df, x='Algoritmo', y='Tempo_Compressao_segundos', hue='Algoritmo', ax=ax3, palette='magma', errorbar=None, legend=False)
    ax3.set_title('GeoTIFF ZSTD: Tempo Médio para Compressão (s)', fontsize=12, fontweight='bold')
    ax3.set_xlabel('')
    ax3.set_ylabel('Tempo (s)')
    ax3.tick_params(axis='x', rotation=45)

    # 4. Tempo Médio de Leitura após Compressão (s)
    ax4 = axes[1, 1]
    sns.barplot(data=df, x='Algoritmo', y='Tempo_Leitura_segundos', hue='Algoritmo', ax=ax4, palette='mako', errorbar=None, legend=False)
    ax4.set_title('GeoTIFF ZSTD: Tempo Médio de Leitura (s)', fontsize=12, fontweight='bold')
    ax4.set_xlabel('')
    ax4.set_ylabel('Tempo (s)')
    ax4.tick_params(axis='x', rotation=45)

    # Adiciona rótulos com os valores numéricos em cada barra
    for ax in axes.flat:
        for container in ax.containers:
            ax.bar_label(container, fmt='%.3f', padding=3, fontsize=9)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'graficos_gtiff_zstd.png', dpi=150)
    plt.close()

    df_resumo = df.groupby('Algoritmo').agg(
        Tamanho_Medio_MB=('Tamanho_Final_MB', 'mean'),
        Taxa_Media_Compressao=('Taxa_Compressao (%)', 'mean'),
        Tempo_Medio_Compressao_s=('Tempo_Compressao_segundos', 'mean'),
        Tempo_Medio_Leitura_s=('Tempo_Leitura_segundos', 'mean')
    ).reset_index().round(3)

    print('\n' + '=' * 20 + ' RESUMO DE MÉDIAS: GTIFF ZSTD ' + '=' * 20)
    print(df_resumo.to_string(index=False))
    print('=' * 65)

In [ ]:
# Comando principal que executa as funções principais;

In [9]:
if __name__ == '__main__':
    executar_fase_processamento()
    executar_fase_analise()

Conectando ao STAC: https://data.inpe.br/bdc/stac/v1/...

Processando Cena: S2-16D_V2_002011_20260728
--> Métricas GeoTIFF ZSTD para Banda: B01
   [MÉTRICAS SALVAS] metricas_S2-16D_V2_002011_20260728_B01.json
--> Métricas GeoTIFF ZSTD para Banda: B02
   [MÉTRICAS SALVAS] metricas_S2-16D_V2_002011_20260728_B02.json
--> Métricas GeoTIFF ZSTD para Banda: B03
   [MÉTRICAS SALVAS] metricas_S2-16D_V2_002011_20260728_B03.json
--> Métricas GeoTIFF ZSTD para Banda: B04
   [MÉTRICAS SALVAS] metricas_S2-16D_V2_002011_20260728_B04.json
--> Métricas GeoTIFF ZSTD para Banda: B05
   [MÉTRICAS SALVAS] metricas_S2-16D_V2_002011_20260728_B05.json
--> Métricas GeoTIFF ZSTD para Banda: B06
   [MÉTRICAS SALVAS] metricas_S2-16D_V2_002011_20260728_B06.json
--> Métricas GeoTIFF ZSTD para Banda: B07
   [MÉTRICAS SALVAS] metricas_S2-16D_V2_002011_20260728_B07.json
--> Métricas GeoTIFF ZSTD para Banda: B08
   [MÉTRICAS SALVAS] metricas_S2-16D_V2_002011_20260728_B08.json
--> Métricas GeoTIFF ZSTD para Banda: B09
